# 05 — Train YOLO26s: Baseline vs Improved · Banana Ripeness Detection

**หน้าที่ของ notebook นี้ (MASTER PROMPT ข้อ 6)**
1. กำหนด 2 experiment ไว้ที่เดียว: **Baseline** (augmentation default ของ ultralytics, `data.yaml`) vs **Improved** (augmentation ตาม Related Work + oversampling, `data_oversampled.yaml`)
2. **Smoke run 1 epoch** → รู้ batch ที่ AutoBatch เลือก / VRAM / เวลาต่อ epoch → ETA ก่อนเริ่มจริง
3. เทรนทั้ง 2 experiment ด้วย `yolo26s.pt`, imgsz 640, **epochs 150** ตาม spec (อ่านจาก `project.yaml`) + early stopping patience 30, batch auto, seed 42, `optimizer="MuSGD"`
   — resume อัตโนมัติจาก `last.pt` ถ้า kernel หลุด, ข้ามถ้าเสร็จแล้ว
4. สรุป best epoch / mAP50 / mAP50-95 บน valid + learning curve เบื้องต้น → `reports/05_experiments.json` + path `best.pt` ลง `configs/project.yaml`

**หมายเหตุ:** งบสูงสุด 150 epoch ตาม spec — การเทรนหยุดเองเมื่อ val mAP50-95 ไม่ดีขึ้น 30 epoch ติดกัน (รอบจริง 2026-09-24 หยุดที่ epoch 72–111) และใช้น้ำหนักจาก epoch ที่ดีที่สุด (best.pt) · รอบก่อนหน้าเคยใช้ 50 epoch แต่ learning curve ยังไม่อิ่มตัว
ข้อควรทราบ: ultralytics ปิด mosaic ที่ epoch = epochs − close_mosaic (= 140/135) ซึ่งมาไม่ถึงเพราะ early stop ก่อน → ทุกการทดลองเทรนด้วย mosaic ตลอด (เงื่อนไขเท่ากันทุกตัว) — ระบุเป็นข้อจำกัดในเล่ม

> kernel: **ML Environment (ds69) + YOLO26** · ต้องรัน nb01–04 ก่อน · **อย่า restart kernel ระหว่างเทรน** (ปิดแท็บเบราว์เซอร์ได้ kernel รันต่อ; ถ้าหลุดจริง Run All ใหม่จะ resume เอง)


In [ ]:
# === CELL 1 === โหลด config + ตรวจ GPU + กำหนด EXPERIMENTS (Baseline / Improved) ไว้ที่เดียวพร้อมเหตุผลทุก augmentation
# ทำอะไร : อ่าน configs/project.yaml (path ของ data yaml จาก nb04), ตั้งค่า train ที่ใช้ร่วมกัน และ augmentation ของแต่ละ experiment
# ทำไม   : spec ข้อ 6 บังคับ (1) เทียบ Baseline vs Improved (2) อธิบายเหตุผลทุก augmentation โดยเฉพาะ hsv ที่ห้ามแรงจนสีกล้วยเพี้ยน
#          (3) ใช้ MuSGD ของ YOLO26 ต้องระบุตรง ๆ เพราะ optimizer='auto' เลือก MuSGD ให้เฉพาะเมื่อ iterations > 10k
import os, sys, json, time, shutil, random, importlib, math
from pathlib import Path
from datetime import datetime, timedelta
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
import ultralytics

PROJECT_ROOT = Path.cwd().resolve()
CFG_PATH = PROJECT_ROOT / "configs" / "project.yaml"
assert CFG_PATH.exists(), f"ไม่พบ {CFG_PATH} → รัน 01_setup_env.ipynb ก่อน"
with open(CFG_PATH, encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)
DIRS = {k: PROJECT_ROOT / v for k, v in CONFIG["dirs"].items()}
SEED = CONFIG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
CLASSES = {int(k): v for k, v in CONFIG["classes"].items()}
NAMES = list(CLASSES.values())
assert "data_yaml" in CONFIG, "ไม่พบ data_yaml ใน project.yaml → รัน 04_split_dataset.ipynb ก่อน"
for k in ("baseline", "improved"):
    assert (PROJECT_ROOT / CONFIG["data_yaml"][k]).exists(), f"ไม่พบ {CONFIG['data_yaml'][k]} → รัน nb04 ใหม่"
sys.path.insert(0, str(DIRS["src"]))
import viz
importlib.reload(viz)
viz.apply_style()

# --- GPU: ห้ามเทรน 150 epochs บน CPU โดยไม่ตั้งใจ ---
QUICK_TEST = os.environ.get("NB05_QUICK_TEST", "0") == "1"    # โหมดทดสอบโค้ด (1 epoch, ภาพเล็ก, ข้อมูล 10%) — ไม่ใช่ผลจริง
if torch.cuda.is_available():
    DEVICE = 0
    props = torch.cuda.get_device_properties(0)
    print(f"[OK] GPU: {props.name} | VRAM {props.total_memory / 1e9:.1f} GB | torch {torch.__version__} | ultralytics {ultralytics.__version__}")
else:
    DEVICE = "cpu"
    assert QUICK_TEST, "ไม่พบ GPU — เทรน 150 epochs บน CPU ใช้เวลาหลายวัน → ตรวจ CELL 4 ของ nb01 ก่อน (ตั้ง NB05_QUICK_TEST=1 เพื่อทดสอบโค้ดเท่านั้น)"
    print(f"[WARN] ไม่มี GPU → QUICK_TEST บน CPU (ผลไม่ใช่ผลจริง)")

# --- ค่าที่ใช้ร่วมกันทั้ง 2 experiment (spec ข้อ 6 + ข้อตกลง 2026-09-20) ---
RUNS = DIRS["runs"] / "detect"
COMMON = dict(
    model=str(DIRS["weights"] / CONFIG["model"]["base_weights"]),   # yolo26s.pt (pretrained COCO)
    imgsz=CONFIG["model"]["imgsz"],          # 640
    epochs=int(CONFIG["model"]["epochs"]),   # 150 ตาม spec (แก้ที่ project.yaml ที่เดียว) — early stopping (patience) จะหยุดเองเมื่ออิ่มตัว
    patience=CONFIG["model"]["patience"],    # 30 ตาม spec
    batch=CONFIG["model"]["batch"],          # -1 = AutoBatch ~60% VRAM
    seed=SEED, deterministic=True,           # reproducible (ultralytics ตั้ง torch deterministic ให้)
    optimizer=CONFIG["model"]["optimizer"],  # "MuSGD" — ต้องระบุตรง ๆ (auto เลือกให้เฉพาะ iterations > 10k)
    device=DEVICE, workers=8, cache="ram",   # cache ภาพใน RAM (train+val ≈ 5 GB จาก 32 GB) เร่ง I/O
    project=str(RUNS), exist_ok=True, pretrained=True, plots=True, verbose=True, val=True,
)
if QUICK_TEST:   # ทดสอบโค้ดเท่านั้น
    COMMON.update(epochs=1, imgsz=320, batch=8, workers=0, cache=False, fraction=0.1, patience=1, plots=False)
    RUNS = DIRS["runs"] / "detect_quicktest"
    COMMON["project"] = str(RUNS)

# --- Augmentation ต่อ experiment — เหตุผลอยู่ใน comment (spec ข้อ 6) ---
AUG_DEFAULT = dict(   # Baseline = ค่า default ของ ultralytics 8.4 ทุกตัว (ระบุชัดเพื่อบันทึกลงรายงาน)
    hsv_h=0.015, hsv_s=0.70, hsv_v=0.40, degrees=0.0, translate=0.1, scale=0.5, shear=0.0, perspective=0.0,
    fliplr=0.5, flipud=0.0, mosaic=1.0, mixup=0.0, copy_paste=0.0, close_mosaic=10,
)
AUG_IMPROVED = dict(
    hsv_h=0.010,      # hue = แกนสีของความสุก (เขียว→เหลือง→น้ำตาล) → เลื่อนน้อยที่สุดที่ยังทน white balance ของกล้องต่างกัน
    hsv_s=0.30,       # default 0.70 (×0.3–1.7) แรงเกิน: เหลืองซีดจนคล้าย overripe ซีด / เขียวจางเป็นเหลือง → ลดเหลือ ±30%
    hsv_v=0.40,       # ความสว่าง ±40% คงไว้ — ทนแสงสายพาน/เงา (Paper 2: lighting variation) โดยไม่แตะ hue
    degrees=15.0,     # มุมกล้อง/การวางกล้วยบนสายพานไม่คงที่ (Paper 2: camera angle)
    translate=0.1, scale=0.5, shear=0.0, perspective=0.0,   # ระยะกล้องต่างกัน (คง default)
    fliplr=0.5, flipud=0.5,   # กล้วยไม่มี "บน-ล่าง" ที่มีความหมาย → พลิกได้ทั้งสองแกน (เพิ่มความหลากหลายโดยไม่แตะสี)
    mosaic=1.0,       # ต่อ 4 ภาพ → ฝึกฉากหลายผล/หลายสเกลในภาพเดียว (Paper 1: multi-object)
    mixup=0.0,        # mixup ผสมสี 2 ภาพ = ทำลาย feature สี → ปิด
    copy_paste=0.0,   # ต้องมี segmentation mask (ไม่มีในชุดนี้) → ปิด
    close_mosaic=15,  # ปิด mosaic 15 epoch สุดท้าย ให้โมเดลปรับเข้ากับภาพจริงที่ไม่ได้ต่อกัน
)
EXPERIMENTS = {
    "baseline": {"data": str(PROJECT_ROOT / CONFIG["data_yaml"]["baseline"]), "aug": AUG_DEFAULT,
                 "desc": "yolo26s.pt + ultralytics default augmentation + train split ตามจริง"},
    "improved": {"data": str(PROJECT_ROOT / CONFIG["data_yaml"]["improved"]), "aug": AUG_IMPROVED,
                 "desc": "yolo26s.pt + colour-safe HSV + geometric robustness (Paper 2) + mosaic (Paper 1) + oversampled train (imbalance 2.41→1.15)"},
}
RUN = {"notebook": "05_train_yolo26s", "started_at": datetime.now().isoformat(timespec="seconds"), "quick_test": QUICK_TEST,
       "common": {k: v for k, v in COMMON.items()}, "experiments": {}}
print(f"[OK] epochs={COMMON['epochs']} imgsz={COMMON['imgsz']} batch={COMMON['batch']} optimizer={COMMON['optimizer']} seed={SEED} | runs → {RUNS.relative_to(PROJECT_ROOT)}")
print("     Baseline vs Improved (ค่าที่ต่างกัน):")
for k in AUG_DEFAULT:
    if AUG_DEFAULT[k] != AUG_IMPROVED[k]:
        print(f"       {k:<12} {AUG_DEFAULT[k]!s:>6} → {AUG_IMPROVED[k]!s:<6}")
print(f"       data         data.yaml → data_oversampled.yaml")


In [ ]:
# === CELL 2 === Smoke run 1 epoch (config Improved) → batch จริงจาก AutoBatch, VRAM, เวลาต่อ epoch, ETA ของทั้ง 2 experiment
# ทำอะไร : เทรน 1 epoch ลง runs/detect/_smoke แล้วลบทิ้ง — อ่านเวลาจาก results.csv (คอลัมน์ time) และ batch จาก trainer
# ทำไม   : รู้ก่อนว่า pipeline ทั้งหมด (dataset → dataloader → MuSGD → val) ทำงาน และใช้เวลาเท่าไร ก่อนจะปล่อยรันหลายชั่วโมง
smoke_dir = RUNS / "_smoke"
shutil.rmtree(smoke_dir, ignore_errors=True)
model = YOLO(COMMON["model"])
t0 = time.time()
_ = model.train(**{**COMMON, **EXPERIMENTS["improved"]["aug"], "data": EXPERIMENTS["improved"]["data"],
                   "name": "_smoke", "epochs": 1, "patience": 1, "plots": False, "close_mosaic": 0})
wall = time.time() - t0
batch_used = int(getattr(model.trainer, "batch_size", COMMON["batch"]))
vram_gb = torch.cuda.max_memory_reserved() / 1e9 if DEVICE != "cpu" else 0.0
res = pd.read_csv(smoke_dir / "results.csv")
res.columns = [c.strip() for c in res.columns]
epoch_sec = float(res["time"].iloc[-1]) if "time" in res.columns else wall
n_train_baseline = sum(1 for _ in (PROJECT_ROOT / CONFIG["data_yaml"]["baseline"]).parent.joinpath("train", "images").iterdir())
n_train_improved = CONFIG["datasets"]["actual_split"]["train_oversampled_entries"]
eta_improved = epoch_sec * COMMON["epochs"]
eta_baseline = eta_improved * n_train_baseline / max(n_train_improved, 1)
print(f"\n[SMOKE] batch (AutoBatch) = {batch_used} | VRAM สูงสุด {vram_gb:.1f} GB | 1 epoch (train+val) = {epoch_sec:.0f} s | wall รวม setup = {wall:.0f} s")
print(f"[ETA]   Improved ({n_train_improved:,} รายการ/epoch) ≈ {timedelta(seconds=int(eta_improved))} | "
      f"Baseline ({n_train_baseline:,} ภาพ/epoch) ≈ {timedelta(seconds=int(eta_baseline))} | รวม ≈ {timedelta(seconds=int(eta_improved + eta_baseline))}"
      f"  (ไม่รวม early stopping)")
RUN["smoke"] = {"batch": batch_used, "vram_gb": round(vram_gb, 2), "epoch_sec": round(epoch_sec, 1),
                "eta_baseline_min": round(eta_baseline / 60, 1), "eta_improved_min": round(eta_improved / 60, 1)}
shutil.rmtree(smoke_dir, ignore_errors=True)
del model
if DEVICE != "cpu":
    torch.cuda.empty_cache()


In [ ]:
# === CELL 3 === ฟังก์ชันเทรน (resume/skip อัตโนมัติ) + เทรน Baseline
# ทำอะไร : ถ้า runs/detect/<name>/weights/last.pt เป็น checkpoint ที่เทรนจบแล้ว → ข้าม; ถ้าค้างกลางทาง → resume; ไม่มีเลย → เริ่มใหม่
#          บันทึก args ที่ใช้จริง + เวลา + ผล val สุดท้ายลง RUN
# ทำไม   : เทรนหลายชั่วโมง kernel อาจหลุด — ต้อง Run All ซ้ำได้โดยไม่เสียงานเดิม (idempotent) และทุก experiment ต้องมี log ครบ (spec ข้อ 6)
def ckpt_epoch(pt: Path):
    # อ่าน epoch ใน checkpoint: ultralytics ตั้ง epoch = -1 เมื่อเทรนจบ (strip optimizer แล้ว)
    try:
        ck = torch.load(pt, map_location="cpu", weights_only=False)
        return int(ck.get("epoch", 0))
    except Exception:
        return 0

def run_experiment(name: str):
    exp = EXPERIMENTS[name]
    run_dir = RUNS / name
    last, best, res_csv = run_dir / "weights" / "last.pt", run_dir / "weights" / "best.pt", run_dir / "results.csv"
    print(f"\n{'=' * 90}\n[{name.upper()}] {exp['desc']}\n{'=' * 90}")
    t0 = time.time()
    if last.exists() and best.exists() and ckpt_epoch(last) == -1:
        print(f"[SKIP] {run_dir.relative_to(PROJECT_ROOT)} เทรนจบแล้ว (last.pt epoch=-1) → ใช้ผลเดิม")
        results = None
    elif last.exists():
        print(f"[RESUME] ต่อจาก {last.relative_to(PROJECT_ROOT)} (epoch {ckpt_epoch(last)})")
        model = YOLO(str(last))
        results = model.train(resume=True)
    else:
        model = YOLO(COMMON["model"])
        results = model.train(**{**COMMON, **exp["aug"], "data": exp["data"], "name": name})
    train_min = (time.time() - t0) / 60
    assert best.exists() and res_csv.exists(), f"ไม่พบ best.pt/results.csv ใน {run_dir} — การเทรนล้มเหลว ดู error ด้านบน"
    df_res = pd.read_csv(res_csv)
    df_res.columns = [c.strip() for c in df_res.columns]
    m50, m5095 = "metrics/mAP50(B)", "metrics/mAP50-95(B)"
    best_i = int(df_res[m5095].idxmax())
    with open(run_dir / "args.yaml", encoding="utf-8") as f:
        args_used = yaml.safe_load(f)
    rec = {"run_dir": run_dir.relative_to(PROJECT_ROOT).as_posix(), "best": best.relative_to(PROJECT_ROOT).as_posix(), "last": last.relative_to(PROJECT_ROOT).as_posix(),
           "data": exp["data"], "desc": exp["desc"], "epochs_run": int(len(df_res)), "epochs_planned": COMMON["epochs"],
           "best_epoch": int(df_res.loc[best_i, "epoch"]), "val_mAP50": round(float(df_res.loc[best_i, m50]), 4),
           "val_mAP50_95": round(float(df_res.loc[best_i, m5095]), 4),
           "val_precision": round(float(df_res.loc[best_i, "metrics/precision(B)"]), 4), "val_recall": round(float(df_res.loc[best_i, "metrics/recall(B)"]), 4),
           "train_minutes_this_session": round(train_min, 1), "total_train_seconds": float(df_res["time"].iloc[-1]) if "time" in df_res.columns else None,
           "batch": args_used.get("batch"), "optimizer": args_used.get("optimizer"), "aug": {k: args_used.get(k) for k in AUG_DEFAULT},
           "early_stopped": int(len(df_res)) < COMMON["epochs"]}
    RUN["experiments"][name] = rec
    shutil.copy2(res_csv, DIRS["reports"] / f"05_results_{name}.csv")
    print(f"[DONE] {name}: {rec['epochs_run']} epochs ({'early stop' if rec['early_stopped'] else 'ครบ'}) | best epoch {rec['best_epoch']} | "
          f"val mAP50 {rec['val_mAP50']:.4f} | mAP50-95 {rec['val_mAP50_95']:.4f} | P {rec['val_precision']:.3f} R {rec['val_recall']:.3f} | "
          f"เวลารอบนี้ {train_min:.1f} นาที")
    return rec

rec_base = run_experiment("baseline")


In [ ]:
# === CELL 4 === เทรน Improved
# ทำอะไร : experiment ที่ 2 ด้วยฟังก์ชันเดียวกัน (resume/skip อัตโนมัติ)
# ทำไม   : เทียบผลกับ Baseline บน valid/test ชุดเดียวกัน seed เดียวกัน — ต่างกันเฉพาะ augmentation + oversampling
rec_imp = run_experiment("improved")


In [ ]:
# === CELL 5 === สรุปผลบน valid + learning curve เบื้องต้น (ฉบับเต็ม + วินิจฉัย overfit อยู่ nb06)
# ทำอะไร : ตาราง Baseline vs Improved (best epoch, mAP50, mAP50-95, P, R, เวลา) + กราฟ mAP50-95 / box_loss / cls_loss ตาม epoch
# ทำไม   : ต้องเห็นทันทีว่า Improved ดีกว่าจริงไหมก่อนไป nb06 และตัวเลขนี้คือ "ผลบน valid" (ยังไม่ใช่ test)
rows = []
for name, rec in RUN["experiments"].items():
    rows.append({"experiment": name, "epochs": rec["epochs_run"], "best_epoch": rec["best_epoch"], "val_mAP50": rec["val_mAP50"],
                 "val_mAP50-95": rec["val_mAP50_95"], "precision": rec["val_precision"], "recall": rec["val_recall"],
                 "train_min": round((rec["total_train_seconds"] or 0) / 60, 1), "batch": rec["batch"]})
summary = pd.DataFrame(rows).set_index("experiment")
print("— ผลบน valid (best.pt = epoch ที่ mAP50-95 สูงสุด) —")
print(summary)
if {"baseline", "improved"} <= set(summary.index):
    d = summary.loc["improved", "val_mAP50-95"] - summary.loc["baseline", "val_mAP50-95"]
    print(f"\nImproved − Baseline: mAP50-95 {d:+.4f} | mAP50 {summary.loc['improved', 'val_mAP50'] - summary.loc['baseline', 'val_mAP50']:+.4f}")

curves = {name: pd.read_csv(DIRS["reports"] / f"05_results_{name}.csv") for name in RUN["experiments"]}
for c in curves.values():
    c.columns = [k.strip() for k in c.columns]
colors = {"baseline": "#9a9993", "improved": viz.SOURCE_COLORS["mendeley"]}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for name, c in curves.items():
    axes[0].plot(c["epoch"], c["metrics/mAP50-95(B)"], color=colors[name], lw=2, label=name)
    axes[1].plot(c["epoch"], c["train/box_loss"], color=colors[name], lw=2, label=f"{name} train")
    axes[1].plot(c["epoch"], c["val/box_loss"], color=colors[name], lw=1.4, ls="--", label=f"{name} val")
    axes[2].plot(c["epoch"], c["train/cls_loss"], color=colors[name], lw=2, label=f"{name} train")
    axes[2].plot(c["epoch"], c["val/cls_loss"], color=colors[name], lw=1.4, ls="--", label=f"{name} val")
axes[0].set_title("val mAP50-95 per epoch", loc="left")
axes[1].set_title("box loss (solid train, dashed val)", loc="left")
axes[2].set_title("cls loss (solid train, dashed val)", loc="left")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend(fontsize=8)
fig.suptitle(f"Training curves — yolo26s, imgsz {COMMON['imgsz']}, {COMMON['epochs']} epochs, MuSGD, seed {SEED}", x=0.01, ha="left", fontsize=12)
fig.tight_layout()
out = viz.save_fig(fig, DIRS["figures"], "05_training_curves_quick.png")
plt.show()
plt.close(fig)
print(f"[OK] {out.relative_to(PROJECT_ROOT)}")


In [ ]:
# === CELL 6 === บันทึก reports/05_experiments.json + path best.pt ลง project.yaml + สรุป
# ทำอะไร : เก็บ args/ผล/เวลาของทุก experiment และชี้ best.pt ให้ nb06 (evaluate) และ nb07 (video) ใช้
# ทำไม   : nb06/07 ต้องไม่ hardcode path ของ weights — อ่านจาก project.yaml ที่เดียว
RUN["finished_at"] = datetime.now().isoformat(timespec="seconds")
with open(DIRS["reports"] / "05_experiments.json", "w", encoding="utf-8") as f:
    json.dump(RUN, f, ensure_ascii=False, indent=2, default=str)
CONFIG["experiments"] = {name: {"best": rec["best"], "last": rec["last"], "run_dir": rec["run_dir"], "data": rec["data"],
                                "epochs_run": rec["epochs_run"], "best_epoch": rec["best_epoch"],
                                "val_mAP50": rec["val_mAP50"], "val_mAP50_95": rec["val_mAP50_95"]}
                         for name, rec in RUN["experiments"].items()}
CONFIG["model"]["epochs_actual"] = COMMON["epochs"]
with open(CFG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(CONFIG, f, allow_unicode=True, sort_keys=False)
print(f"[OK] reports/05_experiments.json | project.yaml → experiments.{{baseline,improved}}.best")
print("\n=== SUMMARY · Notebook 05 ===")
for name, rec in RUN["experiments"].items():
    print(f"  {name:<9}: {rec['best']}  | val mAP50 {rec['val_mAP50']:.4f} / mAP50-95 {rec['val_mAP50_95']:.4f} (epoch {rec['best_epoch']}/{rec['epochs_run']})")
print("  ถัดไป: 06_evaluate.ipynb (test set, confusion matrix, learning curves, occlusion & robustness, NMS vs NMS-free, speed)")


## ผลลัพธ์ที่ควรเห็น
- CELL 2: `[SMOKE] batch (AutoBatch) = N | VRAM ... | 1 epoch = ... s` และ `[ETA]` — ถ้า ETA รวมเกินที่รับได้ ให้หยุดตรงนี้แล้วปรึกษา
- CELL 3–4: log ของ ultralytics ต่อ epoch (box_loss / cls_loss / dfl_loss / mAP) → จบด้วย `[DONE] baseline: ...` และ `[DONE] improved: ...`
- CELL 5: ตาราง valid + กราฟ — **ตัวเลขนี้เป็นผลบน valid** ผล test ที่ใช้ในรายงานอยู่ nb06
- ไฟล์: `runs/detect/{baseline,improved}/` (weights/best.pt, results.csv, กราฟของ ultralytics), `reports/05_results_*.csv`, `reports/05_experiments.json`

## ปัญหาที่อาจเจอ + วิธีแก้
| อาการ | สาเหตุ | วิธีแก้ |
|---|---|---|
| CUDA out of memory | AutoBatch เลือกใหญ่ไป / มีคนใช้ GPU ร่วม | ตั้ง `COMMON["batch"] = 16` แล้ว Run All (resume จะทำต่อ) |
| kernel หลุดกลางทาง | JupyterHub ตัด session | Run All ใหม่ — CELL 3/4 จะ `[RESUME]` จาก last.pt เอง |
| `[SKIP] ... เทรนจบแล้ว` ทั้งที่อยากเทรนใหม่ | มี run เดิมอยู่ | ลบโฟลเดอร์ `runs/detect/<name>` แล้ว Run All |
| RAM ไม่พอตอน cache | เซิร์ฟเวอร์ใช้ RAM ร่วมกัน | ตั้ง `COMMON["cache"] = False` (ช้าลงเล็กน้อย) |
| `deterministic` เตือน/ช้า | บาง op บน GPU ไม่มี deterministic kernel | ปกติ — ultralytics จัดการเอง; ถ้า error ให้ตั้ง `deterministic=False` (ผลจะซ้ำได้ไม่ 100%) |
